# JavaScript & TypeScript in DataX

DataX runs a full JavaScript engine alongside Python and R.
This notebook demonstrates the most useful JS/TS capabilities:

- Basic JS execution and output
- TypeScript (compiled automatically)
- Node.js-compatible modules (`node:path`, `node:fs/promises`)
- Interactive widgets with anywidget


## 1. JavaScript Basics

In [1]:
%%js
const langs = ["Python", "R", "JavaScript", "TypeScript"];

langs.forEach((lang, i) => {
    console.log(`${i + 1}. Hello from ${lang}!`);
});

const result = langs.map(l => l.length).reduce((a, b) => a + b, 0);
console.log(`Total characters: ${result}`);


1. Hello from Python!
2. Hello from R!
3. Hello from JavaScript!
4. Hello from TypeScript!
Total characters: 27


## 2. TypeScript

TypeScript is compiled via esbuild-wasm before execution. Full type checking and modern syntax are supported.

In [2]:
%%ts
interface DataPoint {
    label: string;
    value: number;
}

function summarise(data: DataPoint[]): string {
    const total = data.reduce((sum, d) => sum + d.value, 0);
    const avg   = total / data.length;
    return `Count: ${data.length}, Total: ${total}, Average: ${avg.toFixed(2)}`;
}

const sales: DataPoint[] = [
    { label: "Jan", value: 120 },
    { label: "Feb", value: 95  },
    { label: "Mar", value: 142 },
];

console.log(summarise(sales));
sales.forEach(d => console.log(`  ${d.label}: ${d.value}`));


Count: 3, Total: 357, Average: 119.00
  Jan: 120
  Feb: 95
  Mar: 142


## 3. Node.js-Compatible Modules

DataX ships shims for common Node.js built-ins so you can use
`node:path` and `node:fs/promises` even when running in the browser.

In [3]:
%%js
import path from "node:path";

const full = "/home/user/projects/analysis/data.csv";
console.log("basename:", path.basename(full));        // data.csv
console.log("dirname: ", path.dirname(full));         // /home/user/projects/analysis
console.log("extname: ", path.extname(full));         // .csv
console.log("join:    ", path.join("projects", "2024", "report.md"));


basename: data.csv
dirname:  /home/user/projects/analysis
extname:  .csv
join:     projects/2024/report.md


## 4. Async / Await

Async functions work exactly as in Node.js.

In [4]:
%%js
async function fetchExample() {
    // Simulated async operation
    const data = await Promise.resolve({ status: "ok", items: [1, 2, 3] });
    console.log("Status:", data.status);
    console.log("Items: ", JSON.stringify(data.items));
    return data;
}

fetchExample();


Status: ok
Items:  [1,2,3]


## 5. Interactive JavaScript

In [5]:
%%js
function renderTicTacToe({ model, el }) {
  const board = model.get("board") || Array(9).fill("");
  const currentPlayer = model.get("current_player") || "X";
  const winner = model.get("winner") || "";

  if (!model.get("board")) {
    model.set("board", board);
    model.set("current_player", currentPlayer);
    model.set("winner", winner);
    model.save_changes();
  }

  const container = document.createElement("div");
  container.style.fontFamily = "system-ui, sans-serif";
  container.style.maxWidth = "280px";
  container.style.display = "grid";
  container.style.gap = "12px";

  const status = document.createElement("div");
  status.style.fontWeight = "700";
  container.appendChild(status);

  const grid = document.createElement("div");
  grid.style.display = "grid";
  grid.style.gridTemplateColumns = "repeat(3, 72px)";
  grid.style.gap = "6px";
  container.appendChild(grid);

  const reset = document.createElement("button");
  reset.textContent = "New game";
  reset.type = "button";
  container.appendChild(reset);
  el.appendChild(container);

  const winningLines = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
  ];

  function getWinner(values) {
    for (const [a, b, c] of winningLines) {
      if (values[a] && values[a] === values[b] && values[a] === values[c]) {
        return values[a];
      }
    }
    return values.every(Boolean) ? "draw" : "";
  }

  function render() {
    const values = model.get("board") || Array(9).fill("");
    const player = model.get("current_player") || "X";
    const gameWinner = model.get("winner") || getWinner(values);
    status.textContent = gameWinner
      ? gameWinner === "draw" ? "Draw" : `${gameWinner} wins!`
      : `Turn: ${player}`;
    while (grid.firstChild) grid.removeChild(grid.firstChild);

    values.forEach((value, index) => {
      const cell = document.createElement("button");
      cell.type = "button";
      cell.textContent = value;
      cell.style.width = "72px";
      cell.style.height = "72px";
      cell.style.fontSize = "32px";
      cell.style.fontWeight = "700";
      cell.disabled = Boolean(value || gameWinner);
      cell.addEventListener("click", () => {
        const nextBoard = [...(model.get("board") || Array(9).fill(""))];
        if (nextBoard[index] || model.get("winner")) return;
        nextBoard[index] = model.get("current_player") || "X";
        const nextWinner = getWinner(nextBoard);
        model.set("board", nextBoard);
        model.set("winner", nextWinner);
        if (!nextWinner) {
          model.set("current_player", (model.get("current_player") || "X") === "X" ? "O" : "X");
        }
        model.save_changes();
      });
      grid.appendChild(cell);
    });
  }

  reset.addEventListener("click", () => {
    model.set("board", Array(9).fill(""));
    model.set("current_player", "X");
    model.set("winner", "");
    model.save_changes();
    render();
  });

  model.on("change", render);
  render();
}

display_widget(datax.widget({
  model: { board: Array(9).fill(""), current_player: "X", winner: "" },
  createView: renderTicTacToe
}));

A Jupyter Widget

## Summary

| Feature | Magic | Notes |
|---------|-------|-------|
| JavaScript | `%%js` | Full ES2022+ |
| TypeScript | `%%ts` / `%%typescript` | Compiled via esbuild |
| Node modules | `import x from "node:..."` | `path`, `fs/promises` shims |
| Widgets | Python `anywidget` | Bidirectional state sync |
